# SNR interval figure

Builds the schematic SNR-interval illustration from one selected waveform window.


In [ ]:
# ==========================================
# 1. IMPORTS AND PATHS
# ==========================================
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator, MaxNLocator
from obspy.signal.trigger import classic_sta_lta


def _find_repo_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "data" / "raw").is_dir() and (p / "src").is_dir():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
# Set REPO_ROOT explicitly if automatic detection fails.
# REPO_ROOT = Path(r"M:\Github\TFM_SeismicPhasePicker_ColombianAndesRegion")

sys.path.insert(0, str(REPO_ROOT / "src" / "preprocessing"))
import preprocessing_utils as pp  # noqa: E402

WAVEFORMS_DIR = REPO_ROOT / "data" / "raw" / "waveforms"
FIGURES_DIR   = REPO_ROOT / "data" / "processed" / "images" / "theory"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Figuras ->", FIGURES_DIR)


In [ ]:
# ==========================================
# 2. CONFIGURATION
# ==========================================
STATION   = "ZAR"
TIMESTAMP = "2018.344.11.50.07"   # record used in the P/S illustration

SNR_WIN_S = 5.0     # measurement interval, seconds

# Estimate P with STA/LTA; use the selected S marker from
# data_preprocessing_figures.ipynb.
STA_P_S, LTA_P_S = 1.0, 10.0
P_SEARCH_S = (22.0, 36.0)
STALTA_THR = 3.0
S_ARRIVAL_S = 42.08

# Figure style
PROC_COLOR   = "crimson"
P_COLOR      = "k"
S_COLOR      = "#1f77b4"
NOISE_COLOR  = "#8c8c8c"
SIGNAL_COLOR = "#1f4e9c"

DPI = 300
SAVE_FORMATS = ["pdf", "png"]
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"]  = 42
plt.rcParams["font.size"] = 12

sta_dir = WAVEFORMS_DIR / STATION
fs = pp.SAMPLING_RATE_HZ


def save_fig(fig, name):
    for fmt in SAVE_FORMATS:
        out = FIGURES_DIR / f"{name}.{fmt}"
        fig.savefig(out, dpi=DPI, bbox_inches="tight", pad_inches=0.08)
        print("guardado", out)


In [ ]:
# ==========================================
# 3. P ARRIVAL (STA/LTA ON Z)
# ==========================================
arr = pp.load_window_array(sta_dir, TIMESTAMP, preprocess=True)
if arr is None:
    raise RuntimeError(f"{TIMESTAMP}: la ventana no superó el control de calidad.")

t = np.arange(pp.NPTS) / fs
z, n, e = arr[0], arr[1], arr[2]


def auto_onset(sig, sta_s, lta_s, lo_s, hi_s, thr=STALTA_THR):
    cft = classic_sta_lta(np.asarray(sig, dtype=float), int(sta_s * fs), int(lta_s * fs))
    lo, hi = int(lo_s * fs), int(hi_s * fs)
    seg = cft[lo:hi]
    above = np.where(seg > thr)[0]
    idx = lo + (int(above[0]) if above.size else int(np.argmax(seg)))
    return idx / fs


p_s = auto_onset(z, STA_P_S, LTA_P_S, *P_SEARCH_S)
s_s = S_ARRIVAL_S
print(f"Onda P (STA/LTA): {p_s:.2f} s")
print(f"Onda S (validada a ojo, ver data_preprocessing_figures.ipynb): {s_s:.2f} s")


In [ ]:
# ==========================================
# 4. FIGURE
# ==========================================
# Show the P interval on Z and the S interval on both horizontal components.
fig, ax = plt.subplots(3, 1, figsize=(10, 6.4), sharex=True)
labels = {"Z": "Amplitud\n(componente Z)",
          "N": "Amplitud\n(componente N)",
          "E": "Amplitud\n(componente E)"}
data = {"Z": z, "N": n, "E": e}

for i, comp in enumerate("ZNE"):
    ax[i].plot(t, data[comp], lw=0.7, color=PROC_COLOR, rasterized=True)
    ax[i].axvline(p_s, color=P_COLOR, ls="--", lw=1.2,
                  label="Llegada de la onda P" if i == 0 else None)
    ax[i].axvline(s_s, color=S_COLOR, ls="--", lw=1.2,
                  label="Llegada de la onda S" if i == 0 else None)
    if comp == "Z":
        ax[i].axvspan(p_s - SNR_WIN_S, p_s, color=NOISE_COLOR, alpha=0.35)
        ax[i].axvspan(p_s, p_s + SNR_WIN_S, color=SIGNAL_COLOR, alpha=0.22)
    if comp in ("N", "E"):
        ax[i].axvspan(s_s - SNR_WIN_S, s_s, color=NOISE_COLOR, alpha=0.35)
        ax[i].axvspan(s_s, s_s + SNR_WIN_S, color=SIGNAL_COLOR, alpha=0.22)
    ax[i].set_ylabel(labels[comp])
    ax[i].margins(x=0)
    ax[i].yaxis.set_major_locator(MaxNLocator(nbins=6))

ax[-1].set_xlabel("Tiempo (s)")
ax[0].set_xlim(20, 60)
ax[0].xaxis.set_major_locator(MultipleLocator(10))

handles = [*ax[0].get_legend_handles_labels()[0],
           Patch(facecolor=NOISE_COLOR, alpha=0.35, label="Ventana de ruido (5 s)"),
           Patch(facecolor=SIGNAL_COLOR, alpha=0.22, label="Ventana de señal (5 s)")]
ax[0].legend(handles=handles, loc="upper right", framealpha=0.9, fontsize=9, ncol=1)

plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# 5. SAVE
# ==========================================
# Save after reviewing the figure.
save_fig(fig, "IntervalosSNR_ejemplo")
